#### **Importation des bibliothèques**

In [ ]:
# Importation du module dagshub pour lier le notebook au dépôt distant
import dagshub
# Importation de mlflow pour suivre l'historique et les performances des modèles
import mlflow

# Importation de pandas pour la gestion des données tabulaires (DataFrame)
import pandas as pd
# Importation de 're' pour utiliser les expressions régulières (Regex) lors du nettoyage de texte
# Importation de la librairie emoji pour convertir les symboles visuels en texte compréhensible
# Importation de train_test_split pour diviser notre jeu de données (Train et Test)



#### **DagsHub & MLflow Init**

In [ ]:

# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.
# Ceci corrige l'erreur "charmap codec can't encode characters" causée par le nouveau format d'affichage (summary) de Keras 3
import builtins
_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets_Niveau_3_et_4")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


#### **Modèles Transformers via Hugging Face**

In [ ]:
# Import de l'objet Dataset de Hugging Face
from datasets import Dataset
# Import de evaluate pour calculer de façon standardisée les métriques d'évaluation
# Import de NumPy
import numpy as np

# Transformation de notre DataFrame d'entraînement Pandas en un objet "Dataset" ultra-optimisé de Hugging Face
hf_train = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
# Transformation de notre DataFrame de test Pandas
hf_test = Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))

# Importation de scikit-learn pour calculer facilement le F2-Score et les métriques par classe
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, fbeta_score

# Fonction exécutée à la fin de chaque Epoch par le Trainer pour calculer le score
def compute_metrics(eval_pred):
    # Séparation des probabilités prédites (logits) et des vraies réponses (labels)
    logits, labels = eval_pred
    # L'argmax récupère la classe ayant reçu la plus forte probabilité (0 ou 1)
    predictions = np.argmax(logits, axis=-1)
    
    # Précision et Rappel par classe
    precision_cls = precision_score(labels, predictions, average=None)
    recall_cls = recall_score(labels, predictions, average=None)
    
    # Calcul des métriques globales
    f1 = f1_score(labels, predictions, average="macro")
    f2 = fbeta_score(labels, predictions, beta=2, average="macro")
    accuracy = accuracy_score(labels, predictions)
    
    # Retourner toutes les métriques pour le suivi MLflow (le Trainer ajoutera automatiquement le préfixe "eval_")
    return {
        "f1_macro": f1,
        "f2_score": f2,
        "precision_class_0": precision_cls[0],
        "precision_class_1": precision_cls[1],
        "recall_class_0": recall_cls[0],
        "recall_class_1": recall_cls[1],
        "accuracy": accuracy
    }

# Importation du système d'exploitation
import os
# Paramétrage de la variable d'environnement qui indique à Hugging Face dans quel dossier MLflow il doit écrire
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Disaster_Tweets_Niveau_3_et_4"


In [ ]:
# Import des AutoClasses (la magie de Hugging Face pour importer n'importe quel modèle du web en 1 ligne)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Définition d'une fonction Python réutilisable pour entraîner n'importe quel Transformer sans réécrire le code
def train_hf_model(model_id, run_name, batch_size=16, epochs=2):
    # Affichage du démarrage
    print(f"========== Début de l'entraînement pour {model_id} ==========")
    
    # 1. Chargement du Tokenizer spécifique au modèle (le dictionnaire qui transforme les mots en IDs)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Fonction qui applique le tokenizer sur une phrase
    def tokenize_function(examples):
        # On coupe les phrases (truncation=True) à 128 "tokens" maximum et on ajoute du vide (padding) pour les plus courtes
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)
    
    # Application massive et extrêmement rapide (batched=True) du Tokenizer sur tout le jeu d'entraînement
    tokenized_train = hf_train.map(tokenize_function, batched=True)
    # Même chose pour le test
    tokenized_test = hf_test.map(tokenize_function, batched=True)
    
    # 2. Chargement de l'architecture du Transformer avec une tête de classification pour 2 sorties (0 ou 1)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    
    # 3. Paramètres de l'entraînement 
    training_args = TrainingArguments(
        output_dir=f"./results_{run_name}",  # Dossier de sauvegarde
        eval_strategy="epoch",         # Evaluer le modèle à la fin de chaque Epoch
        save_strategy="epoch",               # Sauvegarder un "point de contrôle" à la fin de chaque Epoch
        learning_rate=2e-5,                  # Taux d'apprentissage très petit (spécifique aux Transformers)
        per_device_train_batch_size=batch_size, # Taille des paquets envoyés à la carte graphique (entraînement)
        per_device_eval_batch_size=batch_size,  # Taille des paquets envoyés à la carte graphique (test)
        num_train_epochs=epochs,             # Nombre total d'itérations
        weight_decay=0.01,                   # Ajout de pénalités pour éviter le surapprentissage
        load_best_model_at_end=True,         # A la fin, on recharge la version qui a eu le meilleur score
        report_to="mlflow",                  # Dit au système d'envoyer tout le suivi de cet entraînement vers MLflow (DagsHub)
        run_name=run_name,                   # Nom du run dans l'interface MLflow
    )
    
    # 4. L'Objet Trainer qui s'occupe de gérer toute la boucle mathématique PyTorch en arrière-plan
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics, # On utilise notre fonction personnalisée pour mesurer le F1-Score
    )
    
    # Démarre l'entraînement intensif
    trainer.train()
    
    try:
        # Tente d'enregistrer le modèle HuggingFace dans MLflow pour la mise en production
        components = {"model": trainer.model, "tokenizer": tokenizer}
        mlflow.transformers.log_model(transformers_model=components, artifact_path="model")
    except Exception as e:
        print("Avertissement: L'enregistrement du modèle Transformers dans MLflow a échoué:", e)
    
    # Force MLflow à fermer proprement la session de suivi de ce run
    mlflow.end_run()
    # Affiche la fin dans la console
    print(f"========== Fin de l'entraînement pour {model_id} ==========\n")


#### **Modèle DeBERTa-v3 (L'État de l'Art Suprême)**

##### **Description du modèle**
Créé par Microsoft, c'est l'un des meilleurs modèles au monde aujourd'hui pour les tâches de classification.

##### **Explication du fonctionnement**
DeBERTa améliore le mécanisme d'attention en utilisant une **Attention Désintriquée (Disentangled Attention)**. Là où BERT mélange le "mot" et sa "position absolue dans la phrase", DeBERTa traite le contenu du mot et sa position relative séparément. De plus, la version v3 utilise une technique spéciale d'apprentissage génératif qui le rend extrêmement robuste.



In [ ]:
# Entraînement de DeBERTa.
# IMPORTANT : J'ai passé l'argument batch_size=8 car ce modèle consomme beaucoup plus de mémoire vidéo (VRAM).
# Sans cette réduction, l'entraînement risque de faire crasher la carte graphique (Erreur OOM : Out Of Memory).
train_hf_model(model_id="microsoft/deberta-v3-base", run_name="4.4_DeBERTa", batch_size=8, epochs=2)
